# Ambient Deployment and L4/L7 Policies Demo

**Solo Enterprise for Istio, ambient mode, on kind.** One app (a petshop), every ambient security feature, and the Solo Enterprise differences called out as we go.

In [ ]:
# Run from the lab directory. Constants used throughout.
[ -d istio-ambient-cert-identity-kind ] && cd istio-ambient-cert-identity-kind || :
export CTX=kind-cert-identity
export ISTIO_NS=istio-system
# license (edit if your secrets live elsewhere)
export SECRETS_FILE="${SECRETS_FILE:-$HOME/code/solo/secrets/secrets-envs.sh}"
[ -f "$SECRETS_FILE" ] && set -a && . "$SECRETS_FILE" && set +a
echo "context: $CTX ; license set: $([ -n "$SOLO_ISTIO_LICENSE_KEY" ] && echo yes || echo NO)"


## 1 · Stand up the mesh

Ambient here is the **four Solo-distribution Helm charts** (`base`, `istiod`, `cni`, `ztunnel`), profile `ambient`


In [ ]:
# 1a. kind cluster (1 control-plane + 1 worker)
kind create cluster --config kind/cluster.yaml


**Kick off the Solo UI now, in the background.** The Gloo Platform management plane (mgmt server, agent, UI, Prometheus, Redis, OTel) has no Istio dependency and takes the longest of anything in §1 — so start it the moment the cluster exists and let it install while we stand up ambient. §2 checks the result and port-forwards. `gloo-platform` 2.13.x pairs with Istio 1.30.


In [ ]:
# 1b. Solo UI (Gloo Platform mgmt plane) — full install, backgrounded to a log.
export GLOO_MESH_NS=gloo-mesh GLOO_PLATFORM_VERSION=2.13.2
(
  helm repo add gloo-platform https://storage.googleapis.com/gloo-platform/helm-charts || true
  helm repo update gloo-platform
  kubectl --context $CTX create namespace $GLOO_MESH_NS --dry-run=client -o yaml | kubectl --context $CTX apply -f -

  helm --kube-context $CTX upgrade -i gloo-platform-crds gloo-platform/gloo-platform-crds \
    -n $GLOO_MESH_NS --version $GLOO_PLATFORM_VERSION --wait

  # register the cluster BEFORE the main install (the CRD is in place now)
  kubectl --context $CTX apply -f - <<'REG'
apiVersion: admin.gloo.solo.io/v2
kind: KubernetesCluster
metadata: { name: cert-identity, namespace: gloo-mesh }
spec: { clusterDomain: cluster.local }
REG

  helm --kube-context $CTX upgrade -i gloo-platform gloo-platform/gloo-platform \
    -n $GLOO_MESH_NS --version $GLOO_PLATFORM_VERSION -f - <<VALUES
common: { cluster: cert-identity }
licensing:
  glooMeshLicenseKey: "${GLOO_PLATFORM_LICENSE_KEY:-$SOLO_ISTIO_LICENSE_KEY}"
glooMgmtServer: { enabled: true, createGlobalWorkspace: true }
glooUi: { enabled: true, serviceType: ClusterIP }
glooAgent:
  enabled: true
  relay: { serverAddress: gloo-mesh-mgmt-server.gloo-mesh:9900 }
prometheus: { enabled: true }
redis: { deployment: { enabled: true } }
telemetryCollector: { enabled: true }
telemetryGateway: { enabled: true }
glooInsightsEngine: { enabled: true }
VALUES
  echo "GLOO INSTALL DONE"
) > /tmp/cert-identity-gloo-install.log 2>&1 &
echo "Solo UI installing in the background — log: /tmp/cert-identity-gloo-install.log"


In [ ]:
# 1c. Gateway API CRDs (used by the waypoint later)
kubectl --context $CTX apply --server-side -f \
  https://github.com/kubernetes-sigs/gateway-api/releases/download/v1.5.1/standard-install.yaml


In [ ]:
# 1d. Pre-pull the Solo Istio images on the host and load them into kind.
#     (docker save | kind load, because kind's containerd chokes on multi-arch indexes.)
REG=us-docker.pkg.dev/soloio-img/istio ; VER=1.30.3-solo
case "$(uname -m)" in arm64|aarch64) PLAT=linux/arm64;; *) PLAT=linux/amd64;; esac
for img in pilot proxyv2 install-cni ztunnel; do
  docker pull -q $REG/$img:$VER
  docker save --platform $PLAT $REG/$img:$VER -o /tmp/$img.tar
  kind load image-archive /tmp/$img.tar --name cert-identity && rm -f /tmp/$img.tar
done


On the 1.30 line the chart version **and** the image tag both keep the `-solo` suffix (`1.30.3-solo`). Keep it on the tag — the plain `1.30.3` image in the same registry is the upstream build, with none of the Solo additions. The trust domain is set directly with `meshConfig.trustDomain: cert-identity`, so identities are `spiffe://cert-identity/ns/<ns>/sa/<sa>` — **not** `cluster.local`.


In [ ]:
# helm repo + versions for the Solo distribution
export HUB=us-docker.pkg.dev/soloio-img/istio TAG=1.30.3-solo
export HREPO=oci://us-docker.pkg.dev/soloio-img/istio-helm HVER=1.30.3-solo

# 1e. base — Istio CRDs + cluster roles
helm --kube-context $CTX upgrade -i istio-base $HREPO/base \
  -n $ISTIO_NS --create-namespace --version $HVER --set defaultRevision=default --wait


In [ ]:
# 1f. istiod (profile ambient) — LICENCE, TRUST DOMAIN and access logs are VALUES, not patches
helm --kube-context $CTX upgrade -i istiod $HREPO/istiod -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
global: { hub: $HUB, tag: $TAG }
istio_cni: { enabled: true }
license: { value: $SOLO_ISTIO_LICENSE_KEY }
env: { PILOT_SKIP_VALIDATE_TRUST_DOMAIN: "true" }
meshConfig:
  accessLogFile: /dev/stdout
  trustDomain: cert-identity
EOF


In [ ]:
# 1g. istio-cni (traffic capture) + ztunnel (per-node L4 proxy).
#     ztunnel gets JSON logs + Solo L7 telemetry as env VALUES — again, no kubectl patch.
helm --kube-context $CTX upgrade -i istio-cni $HREPO/cni -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
global: { hub: $HUB, tag: $TAG }
ambient: { dnsCapture: true }
excludeNamespaces: [istio-system, kube-system]
EOF

helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
env: { LOG_FORMAT: json, L7_ENABLED: "true" }
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel daemonset/istio-cni-node --timeout=180s
echo "ambient ready (Helm, no operator), trust domain cert-identity, JSON access logs on"


**Kick off the IdP now, in the background.** Keycloak takes ~40–60s to deploy.


In [ ]:
kubectl --context $CTX apply -f yaml/40-idp/keycloak.yaml
echo "Keycloak starting in the background — carry on; it'll be up by the JWT step"


## 2 · Watch it in the Solo UI (Gloo UI)

The **Gloo UI** is Solo's own dashboard for Solo Enterprise for Istio. It is the Gloo Platform management plane; on one kind cluster the mgmt server, the agent and the UI are co-located, and once the cluster is registered the agent discovers the ambient mesh. It has been **installing in the background since §1** — check it finished, then port-forward.


In [ ]:
# 2a. the plane has been installing since §1 — confirm the background install,
#     then wait only for the UI (the one thing we port-forward)
for i in $(seq 1 60); do grep -q "GLOO INSTALL DONE" /tmp/cert-identity-gloo-install.log && break; sleep 5; done
grep -q "GLOO INSTALL DONE" /tmp/cert-identity-gloo-install.log \
  || { echo "background install has not finished — last log lines:"; tail -5 /tmp/cert-identity-gloo-install.log; }
kubectl --context $CTX -n gloo-mesh rollout status deploy/gloo-mesh-ui --timeout=300s


Port-forward the UI in the **background**


In [ ]:
# 2b. background port-forward on a FREE random local port (8090 is often taken
#     by another lab). nohup + & returns straight away; logs to /tmp.
#     Re-running this cell replaces the forward rather than stacking another.
pkill -f 'port-forward.*svc/gloo-mesh-ui' 2>/dev/null || true
PORT=$(python3 -c 'import socket;s=socket.socket();s.bind(("",0));print(s.getsockname()[1]);s.close()')
nohup kubectl --context $CTX -n gloo-mesh port-forward svc/gloo-mesh-ui $PORT:8090 \
  > /tmp/cert-identity-gloo-ui-pf.log 2>&1 &
sleep 4
echo "Gloo UI → http://localhost:$PORT"


Open the URL the previous cell printed. Deploy the petshop in the next step and refresh: `petstore`, `storefront`, `analytics` and both `checkout` workloads appear under the `petshop` namespace, and once traffic flows the Observability graph fills in.


## 3 · The petshop app

Not the stock Istio sample — five tiny workloads built for this lab (an inline Python API plus four curl-loop clients), with the ServiceAccount layout chosen to tell the identity story. One namespace, enrolled in ambient with a single label. The cast:

| Workload | ServiceAccount → identity | Role |
|---|---|---|
| `petstore` | `sa/petstore` | the API (`GET /pets`, `DELETE /pets/{id}`) |
| `storefront` | `sa/storefront` | client the L4 policy **allows** |
| `analytics` | `sa/analytics` | client the L4 policy **denies** |
| `checkout-blue` | `sa/checkout` | shares one SA with green … |
| `checkout-green` | `sa/checkout` | … so both present the **same** SVID |

**Where the traffic comes from:** each client runs a `curl http://petstore:8080/pets` loop *every 2 seconds* and prints the HTTP code. The observe-cells below don't send requests themselves — they read the client's last log line (`kubectl logs --tail=1`) to see the current outcome, which is why each policy step does a short `sleep` first (to let the new rule reach ztunnel). A denied client shows `000000` (connection refused), an allowed one `200`.


In [ ]:
# Enrol the namespace in ambient — one label, no restarts.
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: petshop
  labels:
    istio.io/dataplane-mode: ambient
EOF

# Deploy the petshop (manifests in yaml/10-app/ — petstore is the API, the rest are clients)
kubectl --context $CTX apply -f yaml/10-app/
kubectl --context $CTX -n petshop rollout status deploy/petstore deploy/storefront deploy/analytics deploy/checkout-blue deploy/checkout-green --timeout=180s
kubectl --context $CTX -n petshop get pods


## 4 · Identity is the certificate

ztunnel holds one mTLS SVID per workload **identity**, and the identity is the ServiceAccount — the URI SAN is exactly what an L4 policy matches on. So look for what is *missing* below: five pods, but only **four** leaf certs. `checkout-blue` and `checkout-green` never appear by name; they share `sa/checkout`, so ztunnel presents **one** cert for both. That is the ceiling §7 hits, and §9 removes.


In [ ]:
# five pods, but their identity is the ServiceAccount…
kubectl --context $CTX -n petshop get pods \
  -o custom-columns=POD:.metadata.name,SERVICEACCOUNT:.spec.serviceAccountName

# …so the ztunnel on their node holds only FOUR leaf certs — checkout appears once
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificate "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE NAME|petshop" | grep -Ei "name|leaf"


## 5 · Authorize on identity, at L4

One `AuthorizationPolicy`, enforced by ztunnel, no waypoint. It selects `petstore` and allows only the `storefront` identity — so it simultaneously **allows storefront** and **denies analytics and checkout** (ztunnel fails closed: once any ALLOW selects a workload, everything unnamed is denied).

Principals use the trust domain `cert-identity`


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-storefront, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/storefront"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
# each client curls petstore every 2s and prints the HTTP code
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done


**Checked the Gloo UI Graph and the denied workloads still show edges to `petstore`?** The graph is drawn from **completed-request telemetry** over a rolling window (15 minutes by default) — those edges are the last 15 minutes of *pre-policy* 200s still ageing out, not the denied attempts. A workload that is denied at L4 never completes a request, so once its history rolls out of the window it draws nothing (§8 shows the full arc: allowed, warehouse appears; blocked, its edge goes quiet and ages out). The graph shows **what the mesh serves**; the per-connection **allow/deny verdicts** are in the ztunnel access logs — the next section.

**Make the graph react faster for a demo:** the telemetry itself is quick — the collector scrapes ztunnel every 15s and the UI refreshes every 15s, so fresh data is on screen within ~30 seconds. What lingers is history: use the **time-interval picker in the Graph toolbar** (next to Refresh) and drop it to the smallest window — the graph then reshapes within a minute or two of a policy change.


## 6 · Read the L4 access logs

ztunnel logs every connection with the peer SPIFFE identities and the outcome — identity-aware audit at L4, no waypoint. The clients are calling `petstore` continuously, so there are always fresh lines; with §5's `allow-storefront` policy in force you'll see `storefront` **ALLOW** and `analytics` **DENY**, each tagged with its `src.identity`.


In [ ]:
# ztunnel's L4 access logs, projected to who -> what + the outcome.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
kubectl --context $CTX -n $ISTIO_NS logs "$ZT" --tail=400 \
 | grep '"scope":"access"' | grep petshop | tail -8 \
 | jq -rc '{src:(.["src.identity"]//"-"),
            dst:(.["dst.service"]//.["dst.identity"]//"-"),
            dir:(.direction//"-"),
            result:(if (.error//"")=="" then "ALLOW" else ("DENY: "+(.error|tostring)) end)}'


## 7 · The shared-ServiceAccount ceiling

Add `sa/checkout` to the allowed set. Both `checkout-blue` and `checkout-green` get in — and there is **no L4 rule that lets one in and keeps the other out**. The identity *is* the certificate, the certificate is issued per ServiceAccount, and both pods present the same `sa/checkout` SVID. To ztunnel they are literally the same caller.

Blue/green makes the problem easy to see, but the everyday version is worse: any pod that never sets `serviceAccountName` runs as the namespace's **`default`** ServiceAccount. In a namespace where nobody created dedicated SAs, *every* pod shares `sa/default` — one identity for the lot. Write an L4 ALLOW for "the payments app" and you have admitted every workload in that namespace, including the debug pod someone `kubectl run`'d last week. The policy looks precise; the identity underneath it is not.

Two fixes, and this lab shows the second: give every workload its own ServiceAccount as a baseline (as this app does), and where pods legitimately share one — blue/green deployments, horizontally-split variants of the same service — close the gap with **workload claims**, in §9.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: allow-checkout, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/checkout"] } }]
    to:   [{ operation: { ports: ["8080"] } }]
EOF
sleep 10
for d in storefront analytics checkout-blue checkout-green; do
  echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"
done
# -> storefront 200, checkout-blue 200, checkout-green 200, analytics still 000000


## 8 · What else can L4 decide? Namespace, conditions, DENY

Identity is the headline, but ztunnel authorizes on the whole L4 tuple — source **namespace**, source IP, destination **port**, SNI — directly or in a CEL `when` clause, and a `DENY` beats any `ALLOW`. To decide on *where* a caller is, we add a caller in a second namespace, `warehouse`, and walk a full arc you can follow in the Gloo UI Graph: **allow it and watch it appear, then block it and watch it stop.**

Before you run this: drop the Graph's **time-interval picker** (next to Refresh) to the smallest window. The graph draws served requests over that window, so a small window makes both the appearance and the stop visible within a minute or two.


In [ ]:
# A second client, in ANOTHER namespace, so we can decide on WHERE a caller is.
# It has its own identity (spiffe://cert-identity/ns/warehouse/sa/warehouse-svc)
# and calls petstore across namespaces. (Same manifest as yaml/25-l4-surface/00-warehouse.yaml.)
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: v1
kind: Namespace
metadata:
  name: warehouse
  labels:
    istio.io/dataplane-mode: ambient
---
apiVersion: v1
kind: ServiceAccount
metadata: { name: warehouse-svc, namespace: warehouse }
---
apiVersion: apps/v1
kind: Deployment
metadata: { name: warehouse-svc, namespace: warehouse, labels: { app: warehouse-svc } }
spec:
  replicas: 1
  selector: { matchLabels: { app: warehouse-svc } }
  template:
    metadata: { labels: { app: warehouse-svc } }
    spec:
      serviceAccountName: warehouse-svc
      containers:
        - name: client
          image: curlimages/curl:8.14.1
          command: ["/bin/sh","-c"]
          args:
            - |
              while true; do
                code=$(curl -s -o /dev/null -w '%{http_code}' --max-time 3 http://petstore.petshop:8080/pets || echo 000)
                echo "$(date -u +%H:%M:%S) warehouse-svc -> GET petstore.petshop/pets : $code"
                sleep 2
              done
EOF
kubectl --context $CTX -n warehouse rollout status deploy/warehouse-svc --timeout=90s


**Watching in the Gloo UI?** Add `warehouse` to the Graph's namespace picker. The graph draws nodes from **served traffic**, not inventory — warehouse appears ~30s after its first allowed requests land (15s collector scrape + 15s UI refresh), and it will only appear at all once we allow it below.


**Decide on the source namespace — with a CEL `when` clause.** One ALLOW on `petstore` admits callers from **two** namespaces: `petshop` and `warehouse`. The cross-namespace caller is in — and because it now completes requests, `warehouse` draws itself in the Graph.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop", "warehouse"] }]
EOF
sleep 14
echo "petshop:   $(kubectl --context $CTX -n petshop   logs deploy/storefront    --tail=1)"
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"
# both 200 — now refresh the Graph: the warehouse namespace appears, serving petstore


**Now block it — narrow the `when` back to `petshop`.** Same policy, one value removed. `warehouse-svc` fails closed: an ALLOW selects petstore, and no rule matches warehouse any more. In the Graph, the warehouse edge stops serving — its request rate drops to zero and the edge ages out of the (small) window within a minute or two. The definitive real-time view is ztunnel's access log, which names the reason.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-allow-petshop-namespace, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: ALLOW
  rules:
  - to:   [{ operation: { ports: ["8080"] } }]
    when: [{ key: source.namespace, values: ["petshop"] }]
EOF
sleep 14
echo "warehouse: $(kubectl --context $CTX -n warehouse logs deploy/warehouse-svc --tail=1)"

# the deny as ztunnel logged it — FAIL-CLOSED: allow policies exist, none matched
# the source. The policy takes a few seconds to converge, so poll for the line.
for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep warehouse | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))' 


**DENY beats ALLOW.** ztunnel evaluates CUSTOM → DENY → ALLOW, so a `DENY` rule overrides the namespace `ALLOW`. Here `analytics` is in `petshop` (the ALLOW admits it) but a `DENY` on its identity blocks it anyway — a clean carve-out with no change to the broader policy.

From the client both denials are the same `000000` — the difference is in **ztunnel's access log**. The warehouse deny above read `allow policies exist, but none allowed` (fail-closed); this one reads `explicitly denied by: petshop/l4-deny-analytics` — the log names the policy that fired.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata: { name: l4-deny-analytics, namespace: petshop }
spec:
  selector: { matchLabels: { app: petstore } }
  action: DENY
  rules:
  - from: [{ source: { principals: ["cert-identity/ns/petshop/sa/analytics"] } }]
EOF
sleep 14
for d in storefront analytics; do echo "$d: $(kubectl --context $CTX -n petshop logs deploy/$d --tail=1)"; done
# storefront 200 (still allowed), analytics 000000 (denied: DENY > ALLOW)

# same 000000 at the client — but ztunnel's log shows a DIFFERENT deny: this one
# NAMES the policy. Poll for it (the deny takes a few seconds to converge).
for i in $(seq 1 12); do
  LINE=$(kubectl --context $CTX -n $ISTIO_NS logs -l app=ztunnel --tail=600 2>/dev/null | grep "policy rejection" | grep analytics | tail -1)
  [ -n "$LINE" ] && break; sleep 5
done
echo "$LINE" | python3 -c 'import sys,json; l=sys.stdin.read().strip(); d=json.loads(l) if l else {}; print(d.get("src.identity","(no deny line found — the policy may still be converging; re-run this cell)"),"->",d.get("dst.service",""),"\n  error:",d.get("error",""))' 


In [ ]:
# reset the L4-surface policies before closing the gap
kubectl --context $CTX -n petshop delete authorizationpolicy l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found


## 9 · Closing the shared-ServiceAccount gap — workload claims

§7 hit a wall: two pods on one ServiceAccount are indistinguishable at L4 — one SVID between them. **Workload claims** close it, still at L4, still with no waypoint: with `ENABLE_WORKLOAD_CLAIMS=true`, ztunnel requests a certificate **per pod**, istiod embeds claims in it at issuance, and the `AuthorizationPolicy` matches them with CEL. The mesh is already on the 1.30 line, so this is one Helm value on ztunnel. We left the flag off until now on purpose — the shared-cert gap in §4/§7 **is** the story this step closes.


In [ ]:
# ONE new value: ENABLE_WORKLOAD_CLAIMS on ztunnel. Same chart, same version.
helm --kube-context $CTX upgrade -i ztunnel $HREPO/ztunnel -n $ISTIO_NS --version $HVER --wait -f - <<EOF
profile: ambient
hub: $HUB
tag: $TAG
namespace: $ISTIO_NS
istioNamespace: $ISTIO_NS
env:
  LOG_FORMAT: json
  L7_ENABLED: "true"
  ENABLE_WORKLOAD_CLAIMS: "true"    # per-POD certs + claim enforcement
EOF
kubectl --context $CTX -n $ISTIO_NS rollout status daemonset/ztunnel --timeout=180s


**The flag flips certificates mesh-wide.** ztunnel now keys its certificate cache by **pod**, not ServiceAccount — compare §4, where checkout-blue and checkout-green resolved to one shared SVID. Every leaf below names a pod, and `WIT present` says the cert carries a workload identity token with that pod's claims.


In [ ]:
# every workload now holds a PER-POD cert (contrast §4: one per ServiceAccount)
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" | grep -E "CERTIFICATE|checkout"


**Annotate the pod — the claim is embedded in its certificate at issuance.** istiod reads `solo.io.security-claims/<key>` off the pod and bakes it into the cert, alongside auto claims for the workload name, namespace and pod. The SPIFFE URI never changes (still `sa/checkout`); the claims ride alongside it.


In [ ]:
# blue is the gold tier, green is silver — one annotation each, pods roll
kubectl --context $CTX -n petshop patch deploy checkout-blue  -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"gold"}}}}}'
kubectl --context $CTX -n petshop patch deploy checkout-green -p '{"spec":{"template":{"metadata":{"annotations":{"solo.io.security-claims/tier":"silver"}}}}}'
kubectl --context $CTX -n petshop rollout status deploy/checkout-blue deploy/checkout-green --timeout=120s
sleep 5


In [ ]:
# open the certs with openssl and read the claims off the SAN. Each cert gains an
# otherName SAN under Solo's OID (1.3.6.1.4.1.65865.1.1) whose payload is base64url JSON.
ZT=$(kubectl --context $CTX -n $ISTIO_NS get pod -l app=ztunnel \
      --field-selector spec.nodeName=cert-identity-worker -o jsonpath='{.items[0].metadata.name}')
istioctl --context $CTX ztunnel-config certificates "$ZT.$ISTIO_NS" -o json > /tmp/zt-certs.json

for pod in checkout-blue checkout-green; do
  # the leaf cert ztunnel presents for this pod (the JSON wraps the PEM in base64)
  jq -r ".[] | select(.identity | contains(\"$pod\")) | .certChain[0].pem" /tmp/zt-certs.json \
    | base64 -d > /tmp/$pod.pem

  echo "── $pod ──────────────────────────────────────────────"
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -A1 "Subject Alternative Name" | tail -1

  # decode the otherName payload -> the claims JSON (base64url, no padding)
  openssl x509 -in /tmp/$pod.pem -noout -text | grep -o '65865\.1\.1:.*' | cut -d: -f2 \
    | python3 -c 'import sys,base64,json; p=sys.stdin.read().strip(); print(json.dumps(json.loads(base64.urlsafe_b64decode(p+"="*(-len(p)%4))), indent=2))'
done


Same URI SAN on both (`…/sa/checkout`) — but blue's cert says `tier: gold` and green's says `tier: silver`, signed by istiod. Now authorize on it: **swap §5's SA-wide checkout ALLOW for a claims-scoped one** — the same `when` CEL shape as §8, over `source.claims` instead of `source.namespace` (the `/` in the annotation key becomes `.` in the claim key). While the SA-wide ALLOW stands, it admits both pods regardless of claims, so it goes. Fail-closed: a workload with no claim never matches the ALLOW, so grant with ALLOW on annotated workloads rather than DENY on a missing claim.


In [ ]:
# swap the SA-wide ALLOW for the claims-scoped one: gold only, enforced by
# ztunnel, per hop, no waypoint. Any broader ALLOW would admit green regardless
# of claims, so §5's SA-wide ALLOW and §8's namespace ALLOW both go.
kubectl --context $CTX -n petshop delete authorizationpolicy allow-checkout l4-allow-petshop-namespace l4-deny-analytics --ignore-not-found
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: security.istio.io/v1
kind: AuthorizationPolicy
metadata:
  name: allow-gold-checkout
  namespace: petshop
spec:
  selector:
    matchLabels: { app: petstore }
  action: ALLOW
  rules:
    - from:
        - source:
            principals:
              - cert-identity/ns/petshop/sa/checkout
      to:
        - operation:
            ports: ["8080"]
      when:
        - key: "source.claims['solo.io.security-claims.tier']"
          values: ["gold"]
EOF
kubectl --context $CTX -n petshop get authorizationpolicy allow-gold-checkout -o jsonpath='{.status.conditions[0].message}'; echo; echo

# fresh per-pod certs + the new policy take ~20-30s to converge — wait for both
# outcomes to settle (blue allowed AND green denied)
for i in $(seq 1 24); do
  B=$(kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  G=$(kubectl --context $CTX -n petshop exec deploy/checkout-green -- sh -c \
        'curl -s -o /dev/null -w "%{http_code}" --max-time 3 http://petstore:8080/' 2>/dev/null)
  echo "  waiting for convergence ($i/24): blue=$B green=$G (want blue=200, green!=200)"
  [ "$B" = "200" ] && [ "$G" != "200" ] && break; sleep 5
done
[ "$G" = "200" ] && echo "!! green is still allowed — is another ALLOW policy admitting it? kubectl -n petshop get authorizationpolicy"


for pod in checkout-blue checkout-green; do
  kubectl --context $CTX -n petshop exec deploy/$pod -- sh -c \
    "curl -s -o /dev/null -w '$pod -> %{http_code}\n' --max-time 5 http://petstore:8080/ || echo '$pod -> DENIED at L4'"
done


`checkout-blue -> 200`, `checkout-green -> DENIED` — same ServiceAccount, told apart at L4 by the workload's own certificate. That is the exact thing §7 could not do. The policy status reads `attached to ztunnel` — ztunnel itself enforces this, per hop. Note the distinction from what comes next: **this** CEL is over claims in the workload *certificate* at L4; §12's CEL is over the user's *request token* at the L7 waypoint. The auto claims (`istio.io.workload.pod` and friends) are matchable the same way. The policy also lives in `yaml/60-claims/10-allow-gold-checkout.yaml`.


## 10 · Add a waypoint for L7 — and the waypoint is agentgateway

Everything so far was L4 — including §9's claims. HTTP concerns — methods, paths, JWTs, rate limits — need a **waypoint**, and on Solo Enterprise the waypoint data plane is **agentgateway** (GatewayClass `enterprise-agentgateway-waypoint`), not Envoy. The division of labour stays clean: ztunnel keeps proving the caller's SPIFFE identity over HBONE at L4, then hands the connection to agentgateway for L7 — and the workload identity ztunnel proved rides along, available to every policy as `source.identity.*`.

You opt in at the **namespace** level (one waypoint for every service in `petshop`, the common default) or per-service. We reset the L4 policies here so the L7 story stands on its own (in production you'd keep both — defence in depth).

One shared-CRD note before the install: `gloo-platform-crds` (the §2 Gloo UI) and `enterprise-agentgateway-crds` both ship `authconfigs` and `ratelimitconfigs`. Helm refuses to adopt CRDs owned by another release, so hand those two to the agentgateway release first.


In [ ]:
# reset the L4 policies for a clean L7 story (incl. §9's claims policy)
kubectl --context $CTX -n petshop delete authorizationpolicy allow-storefront allow-checkout allow-gold-checkout --ignore-not-found

# Solo Enterprise for agentgateway — CRDs + control plane (~30s).
# The two CRDs shared with gloo-platform-crds move to the agentgateway release first.
export AGW_VERSION=v2026.7.0
for crd in authconfigs.extauth.solo.io ratelimitconfigs.ratelimit.solo.io; do
  kubectl --context $CTX annotate crd $crd \
    meta.helm.sh/release-name=agentgateway-crds \
    meta.helm.sh/release-namespace=agentgateway-system --overwrite 2>/dev/null || true
done
helm --kube-context $CTX upgrade -i agentgateway-crds \
  oci://us-docker.pkg.dev/solo-public/enterprise-agentgateway/charts/enterprise-agentgateway-crds \
  -n agentgateway-system --create-namespace --version $AGW_VERSION --wait --timeout 3m
# clusterName must match the mesh cluster id (ISTIO_META_CLUSTER_ID = Kubernetes here)
# or the waypoint cannot fetch its certificates
helm --kube-context $CTX upgrade -i agentgateway \
  oci://us-docker.pkg.dev/solo-public/enterprise-agentgateway/charts/enterprise-agentgateway \
  -n agentgateway-system --version $AGW_VERSION \
  --set licensing.licenseKey=$AGENTGATEWAY_LICENSE_KEY \
  --set clusterName=Kubernetes --wait --timeout 5m
kubectl --context $CTX get gatewayclass

# the waypoint itself: a Gateway of class enterprise-agentgateway-waypoint …
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: Gateway
metadata:
  name: petshop-waypoint
  namespace: petshop
  labels: { istio.io/waypoint-for: service }
spec:
  gatewayClassName: enterprise-agentgateway-waypoint
  listeners:
  - name: mesh
    port: 15088
    protocol: HTTP
EOF
# … then enrol the whole petshop NAMESPACE to use it (one waypoint for every service)
kubectl --context $CTX label namespace petshop istio.io/use-waypoint=petshop-waypoint --overwrite
sleep 5
kubectl --context $CTX -n petshop rollout status deploy/petshop-waypoint --timeout=150s
# proof it is agentgateway: watch its own request log as the loops flow through
kubectl --context $CTX -n petshop logs deploy/petshop-waypoint --tail=2


**Namespace or per-service?** We enrolled the whole namespace, so this one waypoint serves every service in `petshop` — the common default: fewer proxies, and you keep full policy granularity because every policy below still targets the waypoint. Switch to **per-service** (`kubectl label service petstore … istio.io/use-waypoint=petshop-waypoint`) only when you want to isolate a sensitive service, scale its waypoint independently, or keep its L7 path separate from the rest of the namespace.

The waypoint's own log already shows the loops flowing through it (`gateway=petshop/petshop-waypoint … http.method=GET http.status=200`) — agentgateway logs every request with method, path and status out of the box.


## 11 · An IdP, and a real JWT

Keycloak runs in its own (non-ambient) namespace with a `petshop` realm and two users: **alice** (`role: user`) and **bob** (`role: admin`). We **started it back in §1** so it's already warm — here we just confirm it, then mint a token and read its claims (the issuer and `realm_access.roles` are what the waypoint checks).


In [ ]:
# mint + decode a token (run from a petshop pod; it reaches Keycloak cross-namespace)
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
BOB=$(tok bob)
echo "$BOB" | cut -d. -f2 | { read p; python3 - "$p" <<'PY'
import sys,base64,json
p=sys.argv[1]; p+="="*(-len(p)%4)
d=json.loads(base64.urlsafe_b64decode(p))
print("iss:  ", d["iss"]); print("user: ", d["preferred_username"]); print("roles:", d["realm_access"]["roles"])
PY
}


## 12 · Authorize on the JWT, at L7

Two `EnterpriseAgentgatewayPolicy` objects on the waypoint. **`jwtAuthentication` (mode: Strict)** validates every request's token against Keycloak's JWKS (fetched in-cluster via the `backendRef`) — no token or a bad token is `401`. **`authorization`** then decides with CEL over the validated claims: any valid token may `GET`; `DELETE` additionally requires `realm_access.roles` to contain `admin`. The `matchExpressions` are OR'd — a request passes if any one holds.


In [ ]:
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-jwt, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    jwtAuthentication:
      mode: Strict
      providers:
        - issuer: http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop
          jwks:
            remote:
              backendRef: { name: keycloak, namespace: keycloak, port: 8080 }
              jwksPath: /realms/petshop/protocol/openid-connect/certs
---
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-authz, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    authorization:
      action: Allow
      policy:
        matchExpressions:
          - 'request.method == "GET"'
          - 'request.method == "DELETE" && "admin" in jwt.realm_access.roles'
EOF
sleep 10


In [ ]:
# the matrix: no token, alice (user), bob (admin).
# Note agentgateway answers a MISSING/invalid token with 401 (authentication),
# and a valid token that fails the CEL with 403 (authorization).
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
tok() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=$1 -d password=$1 -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p'; }
call() { kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c "$1" ; }
ALICE=$(tok alice); BOB=$(tok bob)
echo "no token    GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 http://petstore:8080/pets")"
echo "alice       GET  /pets     -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets")"
echo "alice(user) DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets/1")"
echo "bob(admin)  DELETE /pets/1 -> $(call "curl -s -o /dev/null -w '%{http_code}' -m5 -X DELETE -H 'Authorization: Bearer $BOB' http://petstore:8080/pets/1")"


**Look at the Graph now — the denials are RED.** Two things changed with the waypoint in place. First, the background loops (no tokens) are being refused with `401`, and an L7 denial is a real HTTP **response** — so the Graph can finally draw a block: red edges from every client into `petshop-waypoint`. Contrast the L4 sections, where a denial is a reset connection that produces nothing and blocks only ever showed as an edge going quiet. L4 deny = silence; L7 deny = red edge with a status code. Second, `storefront → keycloak` appears — that is the token minting itself (we exec the curl for alice/bob's tokens from the storefront pod), which is ordinary mesh traffic to a service that is not behind the waypoint.


## 12b · Route at the waypoint — canary and header shift

The waypoint is not just a policy point — agentgateway is a standard **Gateway API** data plane, so the same proxy that checks the JWT also routes. Attach an `HTTPRoute` whose `parentRef` is the petstore **Service** (the GAMMA pattern): callers keep calling `petstore:8080`, and the waypoint enforces the route — a **90/10 canary** to a new `petstore-v2`, plus a header shift (`x-beta: true` goes straight to v2). The §12 JWT policy still gates **every** request at the same waypoint, whichever version it lands on — policy and routing compose.


In [ ]:
# petstore-v2: the same app (it reports served_by: <pod-name>), the SAME
# ServiceAccount (routing does not change identity), its own Service.
kubectl --context $CTX apply -f yaml/50-l7/30-petstore-v2.yaml
kubectl --context $CTX -n petshop rollout status deploy/petstore-v2 --timeout=120s


In [ ]:
# the route: parentRef is the petstore SERVICE — this is mesh routing, no ingress
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: gateway.networking.k8s.io/v1
kind: HTTPRoute
metadata:
  name: petstore-split
  namespace: petshop
spec:
  parentRefs:
    - group: ""
      kind: Service
      name: petstore
      port: 8080
  rules:
    - matches:
        - headers:
            - name: x-beta
              value: "true"
      backendRefs:
        - name: petstore-v2
          port: 8080
    - backendRefs:
        - name: petstore
          port: 8080
          weight: 90
        - name: petstore-v2
          port: 8080
          weight: 10
EOF
kubectl --context $CTX -n petshop get httproute petstore-split \
  -o jsonpath='{.status.parents[0].conditions[?(@.type=="Accepted")].status}{" — accepted by Service/"}{.status.parents[0].parentRef.name}'; echo
sleep 10


In [ ]:
# the proof: distribution, header shift, and the JWT policy still in force
KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "— 20 GETs with alice's token: the canary split —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 20); do curl -s -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' | sort | uniq -c

echo "— 5 GETs with x-beta: true -> always v2 —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 5); do curl -s -m5 -H 'Authorization: Bearer $ALICE' -H 'x-beta: true' http://petstore:8080/pets; echo; done" \
  | grep -o '"served_by": "[^"]*"' | sed 's/.*: "//; s/-[a-z0-9]*-[a-z0-9]*"$//' | sort | uniq -c

echo "— no token, even on the beta route —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -o /dev/null -w 'no token -> %{http_code}\n' -m5 -H 'x-beta: true' http://petstore:8080/pets"


`petstore` ~90%, `petstore-v2` ~10%, the header pins v2, and no token is still `401` on every path — one waypoint doing authentication, authorization **and** traffic management. Watch the Gloo UI Graph: `petstore-v2` appears as it starts serving its share. (`yaml/50-l7/30-petstore-v2.yaml` + `40-route-split.yaml`, or `make l7-routing`.)


## 12c · Rate limit — by workload identity

The user's token says who the **user** is; the certificate says who the **workload** is. Here both meet: a rate limit whose CEL condition keys on `source.identity.serviceAccount` — the SPIFFE identity ztunnel proved at L4 — enforced locally in the waypoint (no external rate-limit server). Only `storefront` draws from this bucket; `checkout` presenting the **same user JWT** is untouched.


In [ ]:
# 5 requests/minute for the storefront IDENTITY, everyone else unlimited
kubectl --context $CTX apply -f - <<'EOF'
apiVersion: enterpriseagentgateway.solo.io/v1alpha1
kind: EnterpriseAgentgatewayPolicy
metadata: { name: petshop-ratelimit-storefront, namespace: petshop }
spec:
  targetRefs:
    - { group: gateway.networking.k8s.io, kind: Gateway, name: petshop-waypoint }
  traffic:
    rateLimit:
      conditional:
        - condition: 'source.identity.serviceAccount == "storefront"'
          policy:
            local:
              - requests: 5
                unit: Minutes
EOF
sleep 8

KC=http://keycloak.keycloak.svc.cluster.local:8080/realms/petshop/protocol/openid-connect/token
ALICE=$(kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "curl -s -m10 -d grant_type=password -d client_id=petshop -d username=alice -d password=alice -d scope=openid $KC" \
  | sed -n 's/.*"access_token":"\([^"]*\)".*/\1/p')

echo "— storefront, 8 rapid GETs (alice's token) —"
kubectl --context $CTX -n petshop exec deploy/storefront -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo

echo "— checkout-blue, 8 rapid GETs (the SAME token) —"
kubectl --context $CTX -n petshop exec deploy/checkout-blue -- sh -c \
  "for i in \$(seq 1 8); do curl -s -o /dev/null -w '%{http_code} ' -m5 -H 'Authorization: Bearer $ALICE' http://petstore:8080/pets; done"; echo


`storefront: 200 ×5 then 429 429 429` — `checkout-blue: 200 ×8`, on the **same token**. The rate budget followed the workload's certificate, not the user's JWT. That is the identity thesis of this whole lab, applied to L7 traffic management. (`yaml/50-l7/50-ratelimit.yaml`, or `make ratelimit`.)


In [ ]:
# tidy up — kill any background port-forwards for this cluster, then the cluster
pkill -f 'kubectl --context kind-cert-identity.*port-forward' 2>/dev/null || true
kind delete cluster --name cert-identity
